# Week 4 Day 6: Resample 和 Rolling
## 時間序列重採樣與移動統計完整指南

**學習目標：**
- 掌握 12+ resample 聚合方法
- 學習 15+ rolling 移動視窗應用
- 理解 expanding 累計統計
- 實戰異常檢測和季節性調整

**時間估計：** 3 小時

In [ ]:
# ========== 環境設定 ==========
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# 顯示設定
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}' if abs(x) < 100 else f'{x:,.0f}')

print("✅ 環境設定完成")

---
## Part 1: 環境設置與資料載入

In [ ]:
# 創建時間序列銷售數據
np.random.seed(42)
date_range = pd.date_range('2023-01-01', periods=365, freq='D')

# 生成帶有趨勢和季節性的銷售數據
base_sales = np.linspace(1000, 2000, 365)  # 趨勢
seasonal = 500 * np.sin(np.arange(365) * 2 * np.pi / 365)  # 季節性
noise = np.random.normal(0, 100, 365)  # 隨機波動

sales_data = pd.DataFrame({
    'date': date_range,
    'sales': base_sales + seasonal + noise,
    'units_sold': np.random.randint(50, 200, 365),
    'category': np.random.choice(['A', 'B', 'C'], 365)
})

sales_data = sales_data.set_index('date')
sales_data['sales'] = sales_data['sales'].clip(lower=0)

print("=== 時間序列銷售數據 ===")
print(sales_data.head(10))
print(f"\n數據形狀：{sales_data.shape}")
print(f"時間範圍：{sales_data.index.min()} 至 {sales_data.index.max()}")

---
## Part 2: Resample 重採樣（12 個範例）

### 2.1 基本重採樣

In [ ]:
# 範例 1-2：日 → 周 → 月
print("=== 範例 1-2：時間頻率轉換 ===")

# 周度銷售聚合
weekly = sales_data['sales'].resample('W').sum()
print(f"\n周度銷售額（共 {len(weekly)} 周）：")
print(weekly.head(10))

# 月度銷售聚合
monthly = sales_data['sales'].resample('M').sum()
print(f"\n月度銷售額（共 {len(monthly)} 月）：")
print(monthly)

In [ ]:
# 範例 3-4：季度和年度聚合
print("=== 範例 3-4：季度與年度聚合 ===")

# 季度聚合
quarterly = sales_data['sales'].resample('Q').sum()
print(f"\n季度銷售額：")
print(quarterly)

# 年度聚合（本例只有1年）
annual = sales_data['sales'].resample('Y').sum()
print(f"\n年度銷售額：")
print(annual)

In [ ]:
# 範例 5：多列重採樣
print("=== 範例 5：多列重採樣與不同聚合函數 ===")

monthly_multi = sales_data.resample('M').agg({
    'sales': ['sum', 'mean', 'min', 'max'],
    'units_sold': ['sum', 'mean']
})

monthly_multi.columns = ['銷售額_合計', '銷售額_均值', '銷售額_最低', '銷售額_最高', '銷量_合計', '銷量_均值']
print(monthly_multi)

In [ ]:
# 範例 6-7：OHLC 重採樣（金融常用）
print("=== 範例 6-7：OHLC 重採樣 ===")

# OHLC：Open-High-Low-Close
weekly_ohlc = sales_data['sales'].resample('W').ohlc()
print("\n週度 OHLC 數據：")
print(weekly_ohlc.head())

monthly_ohlc = sales_data['sales'].resample('M').ohlc()
print("\n月度 OHLC 數據：")
print(monthly_ohlc)

In [ ]:
# 範例 9-10：不同的聚合函數
print("=== 範例 9-10：自定義聚合函數 ===")

# 四分位數
weekly_quantile = sales_data['sales'].resample('W').quantile(0.75)
print("\n週度 75% 分位數：")
print(weekly_quantile.head())

# 自定義函數
def coefficient_variation(x):
    """計算變異係數（CV）"""
    return x.std() / x.mean() if x.mean() != 0 else 0

weekly_cv = sales_data['sales'].resample('W').apply(coefficient_variation)
print("\n週度變異係數：")
print(weekly_cv.head())

In [ ]:
# 範例 11-12：偏移和標籤
print("=== 範例 11-12：Resample 標籤和偏移 ===")

# label 參數：決定時間戳位置
monthly_first = sales_data['sales'].resample('M', label='first').sum()
monthly_last = sales_data['sales'].resample('M', label='last').sum()

print("\n使用 label='first'（月初）：")
print(monthly_first.head())

print("\n使用 label='last'（月末）：")
print(monthly_last.head())

# closed 參數：決定區間開閉
weekly_left = sales_data['sales'].resample('W', closed='left').sum()
weekly_right = sales_data['sales'].resample('W', closed='right').sum()

print(f"\nclosed='left' 與 'right' 的差異：")
print(f"第一周 left: {weekly_left.iloc[0]:.2f}, right: {weekly_right.iloc[0]:.2f}")

---
## Part 3: Rolling 移動視窗（15 個範例）

### 3.1 移動平均

In [ ]:
# 範例 1-3：簡單移動平均 (SMA)
print("=== 範例 1-3：簡單移動平均 ===")

sales_data['SMA_7'] = sales_data['sales'].rolling(window=7).mean()
sales_data['SMA_30'] = sales_data['sales'].rolling(window=30).mean()
sales_data['SMA_90'] = sales_data['sales'].rolling(window=90).mean()

print("\n移動平均對比（前 100 天）：")
comparison = sales_data[['sales', 'SMA_7', 'SMA_30', 'SMA_90']].head(100)
print(comparison.tail(10))

# 可視化平滑效果
print("\n移動平均的平滑效果：")
print(f"原始數據標準差：{sales_data['sales'].std():.2f}")
print(f"7日 SMA 標準差：{sales_data['SMA_7'].std():.2f}")
print(f"30日 SMA 標準差：{sales_data['SMA_30'].std():.2f}")
print(f"90日 SMA 標準差：{sales_data['SMA_90'].std():.2f}")

In [ ]:
# 範例 4-5：加權移動平均 (WMA)
print("=== 範例 4-5：加權移動平均 ===")

# 線性加權
weights = np.arange(1, 8)  # [1,2,3,4,5,6,7]
wma_7 = sales_data['sales'].rolling(7).apply(lambda x: np.sum(x * weights) / np.sum(weights), raw=False)

print("\n加權移動平均（最近的數據權重更高）：")
print(wma_7.head(20))

# 指數加權移動平均 (EMA)
ema_7 = sales_data['sales'].ewm(span=7).mean()
ema_30 = sales_data['sales'].ewm(span=30).mean()

print("\nEMA vs SMA 對比：")
comparison_ema = pd.DataFrame({
    'SMA_7': sales_data['SMA_7'],
    'EMA_7': ema_7,
    'EMA_30': ema_30
}).head(20)
print(comparison_ema)

In [ ]:
# 範例 6-7：移動標準差和變異係數
print("=== 範例 6-7：移動波動性分析 ===")

sales_data['rolling_std_7'] = sales_data['sales'].rolling(7).std()
sales_data['rolling_std_30'] = sales_data['sales'].rolling(30).std()

print("\n移動標準差（衡量波動性）：")
volatility = sales_data[['sales', 'rolling_std_7', 'rolling_std_30']].iloc[30:50]
print(volatility)

# 移動變異係數
moving_cv = sales_data['sales'].rolling(30).apply(lambda x: x.std() / x.mean())
print(f"\n移動變異係數（前 40 天）：")
print(moving_cv.head(40))

In [ ]:
# 範例 10-11：移動最大值和最小值
print("=== 範例 10-11：移動極值分析 ===")

sales_data['rolling_max'] = sales_data['sales'].rolling(30).max()
sales_data['rolling_min'] = sales_data['sales'].rolling(30).min()
sales_data['rolling_range'] = sales_data['rolling_max'] - sales_data['rolling_min']

print("\n移動極值（30日窗口）：")
print(sales_data[['sales', 'rolling_min', 'rolling_max', 'rolling_range']].iloc[40:50])

# 高低波動指示
sales_data['pct_from_min'] = (sales_data['sales'] - sales_data['rolling_min']) / sales_data['rolling_range']
print("\n在 30 日範圍內的百分比位置（0=最低，1=最高）：")
print(sales_data['pct_from_min'].iloc[40:50])

---
## Part 4: Expanding 累計統計（8 個範例）

---
## Part 5: 實戰案例

### 案例 1：銷售趨勢分析